# 34. Scikit-Learn Comprehensive Practice & Problem Sets

10 Real-world Machine Learning problems spanning regression, classification, clustering, hyperparameter tuning, and pipeline engineering with complete solutions.

### Problem Catalog:
1. **Multiple Linear Regression with Ridge Regularization and VIF Assessment**
2. **Binary Classification Benchmark: Logistic Regression vs Random Forest**
3. **KNN Decision Boundary Optimization via 5-Fold Stratified CV**
4. **Pruned Decision Tree vs Gradient Boosting Classifier Comparison**
5. **PCA Dimensionality Reduction with 95% Explained Variance Threshold**
6. **K-Means Clustering with Elbow Method and Silhouette Scoring**
7. **Cost-Sensitive Classification: Tuning Probability Decision Thresholds**
8. **Full End-to-End ColumnTransformer Pipeline with SimpleImputer and OneHotEncoder**
9. **Hyperparameter Optimization with RandomizedSearchCV**
10. **Serializing a Production Pipeline and Executing Batch Inference**

## Problem 1: Ridge Regularized Regression on Housing Features

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

housing = fetch_california_housing(as_frame=True)
X_tr, X_te, y_tr, y_te = train_test_split(housing.data, housing.target, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

ridge = Ridge(alpha=50.0).fit(X_tr_sc, y_tr)
print(f"Problem 1: Ridge R² Score: {r2_score(y_te, ridge.predict(X_te_sc)):.4f}")

## Problem 2: Classification Benchmark (Logistic vs Random Forest)

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

cancer = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(cancer.data, cancer.target, test_size=0.25, random_state=42, stratify=cancer.target)

lr = LogisticRegression(max_iter=1000, random_state=42).fit(StandardScaler().fit_transform(X_tr), y_tr)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)

print(f"Problem 2: Logistic Regression Accuracy: {lr.score(StandardScaler().fit(X_tr).transform(X_te), y_te)*100:.2f}%")
print(f"Problem 2: Random Forest Accuracy:       {rf.score(X_te, y_te)*100:.2f}%")

## Problem 3: KNN Optimal K Search via Stratified K-Fold

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

k_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k), X_tr, y_tr, cv=5).mean() for k in range(3, 15, 2)]
best_idx = np.argmax(k_scores)
print(f"Problem 3: Optimal K: {range(3, 15, 2)[best_idx]} with CV Score: {k_scores[best_idx]*100:.2f}%")

## Problem 4: Decision Tree vs Gradient Boosting

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

dt = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr, y_tr)
gb = GradientBoostingClassifier(n_estimators=50, random_state=42).fit(X_tr, y_tr)

print(f"Problem 4: Pruned Tree Accuracy:      {dt.score(X_te, y_te)*100:.2f}%")
print(f"Problem 4: Gradient Boosting Accuracy: {gb.score(X_te, y_te)*100:.2f}%")

## Problem 5: PCA 95% Explained Variance Reduction

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(StandardScaler().fit_transform(cancer.data))
print(f"Problem 5: Reduced features from {cancer.data.shape[1]} to {X_pca.shape[1]} (Retained 95% Variance)")

## Problem 6: K-Means Clustering Silhouette Evaluation

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(cancer.data)
sil = silhouette_score(cancer.data, km.labels_)
print(f"Problem 6: K-Means (K=3) Silhouette Score: {sil:.4f}")

## Problem 7: Cost-Sensitive Classification Decision Thresholds

In [ ]:
y_probs = lr.predict_proba(StandardScaler().fit(X_tr).transform(X_te))[:, 1]
# Lower threshold from 0.50 to 0.30 to catch more positive cases
y_custom_thresh = (y_probs >= 0.30).astype(int)
print(f"Problem 7: Positives at threshold 0.50: {np.sum(lr.predict(StandardScaler().fit(X_tr).transform(X_te)))}")
print(f"Problem 7: Positives at threshold 0.30: {np.sum(y_custom_thresh)}")

## Problem 8: Mixed-Type ColumnTransformer Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

ct = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='mean')), ('sc', StandardScaler())]), [0, 1]),
    ('cat', OneHotEncoder(handle_unknown='ignore'), [2])
])
pipe = Pipeline([('prep', ct), ('clf', LogisticRegression())])
print("Problem 8: Mixed Preprocessor Pipeline assembled successfully!")

## Problem 9: RandomizedSearchCV Optimization

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {'max_depth': randint(2, 8), 'min_samples_split': randint(2, 6)}
rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), param_dist, n_iter=6, cv=3, random_state=42)
rs.fit(X_tr, y_tr)
print(f"Problem 9: Best Randomized Parameters: {rs.best_params_}")

## Problem 10: Model Serialization and Inference Check

In [ ]:
import joblib
joblib.dump(rf, 'rf_test.joblib')
rf_loaded = joblib.load('rf_test.joblib')
assert np.array_equal(rf.predict(X_te[:3]), rf_loaded.predict(X_te[:3]))
os.remove('rf_test.joblib')
print("Problem 10: Model serialized, reloaded, and verified with 100% precision!")